In [6]:
import pandas as pd
import numpy as np

train_df = pd.read_csv("../data/challenge1_train.csv")
test_df = pd.read_csv("../data/challenge1_test.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

train_df.head()

Train shape: (7398, 10)
Test shape: (2467, 9)


,ID,price,bathrooms,bedrooms,square_feet,latitude,longitude,category,has_photo,state
0,2,925,1.0,0,116,47.6160,-122.3275,housing/rent/apartment,Thumbnail,WA
1,3,2475,1.0,0,130,40.7629,-73.9885,housing/rent/apartment,Thumbnail,NY
2,5,1695,1.0,0,190,37.7599,-122.4379,housing/rent/apartment,Thumbnail,CA
3,6,1560,1.0,1,200,35.0847,-77.0609,housing/rent/apartment,Thumbnail,NC
4,7,1560,1.0,1,200,35.0960,-77.0272,housing/rent/apartment,Thumbnail,NC


In [7]:
print("Data types:")
print(train_df.dtypes)

print("\nMissing values:")
print(train_df.isnull().sum())

print("\nDuplicate rows:", train_df.duplicated().sum())

Data types:
ID               int64
price            int64
bathrooms      float64
bedrooms         int64
square_feet      int64
latitude       float64
longitude      float64
category           str
has_photo          str
state              str
dtype: object

Missing values:
ID             0
price          0
bathrooms      0
bedrooms       0
square_feet    0
latitude       0
longitude      0
category       0
has_photo      0
state          0
dtype: int64

Duplicate rows: 0


In [8]:
X = train_df.drop(columns=["price", "ID"])
y = train_df["price"]

print("Features:")
print(X.columns.tolist())

print("\nTarget: price")
print("X shape:", X.shape)
print("y shape:", y.shape)

Features:
['bathrooms', 'bedrooms', 'square_feet', 'latitude', 'longitude', 'category', 'has_photo', 'state']

Target: price
X shape: (7398, 8)
y shape: (7398,)


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)

X_train: (5918, 8)
X_val: (1480, 8)
y_train: (5918,)
y_val: (1480,)


In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = [
    "bathrooms",
    "bedrooms",
    "square_feet",
    "latitude",
    "longitude"
]

categorical_features = [
    "category",
    "has_photo",
    "state"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)

Numerical features: ['bathrooms', 'bedrooms', 'square_feet', 'latitude', 'longitude']
Categorical features: ['category', 'has_photo', 'state']


## Linear Regression Model

Linear Regression is used as a baseline model to predict monthly apartment rental prices. We use the same preprocessing and validation split to evaluate the model using RMSE.

In [11]:

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np

# Linear Regression pipeline
linear_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

# Train model
linear_pipeline.fit(X_train, y_train)

# Predictions
train_pred = linear_pipeline.predict(X_train)
val_pred = linear_pipeline.predict(X_val)

# RMSE
train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
val_rmse = np.sqrt(mean_squared_error(y_val, val_pred))

print("Linear Regression Results")
print(f"Training RMSE: {train_rmse:.2f}")
print(f"Validation RMSE: {val_rmse:.2f}")


Linear Regression Results
Training RMSE: 515.01
Validation RMSE: 549.97


In [12]:

from pathlib import Path

# Save Linear Regression results
results = pd.DataFrame({
    "Model": ["Linear Regression"],
    "Train RMSE": [train_rmse],
    "Validation RMSE": [val_rmse]
})

Path("../results").mkdir(exist_ok=True)

results.to_csv(
    "../results/linear_regression_results.csv",
    index=False
)

display(results)


,Model,Train RMSE,Validation RMSE
0,Linear Regression,515.007949,549.967519
